<img src="../data/zls.svg" width="120" style="float:right; margin: 0 20px 10px 20px;">

# 02 — STT Basics

Goal: upload an audio file, poll for the transcription, and inspect the returned fields.

### Step 1 — Setup

We import our libraries and point to the audio file we'll transcribe.

In [ ]:
import requests
import time
from pathlib import Path
import pandas as pd

BASE_URL = "https://sproochmaschinn.lu"
AUDIO_PATH = Path("../data/sample_audio.wav")

Create a session and confirm the audio file exists before we try to upload it.

In [ ]:
session = requests.post(f"{BASE_URL}/api/session").json()
session_id = session["session_id"]
print("Session created:", session_id)
print("Audio exists:", AUDIO_PATH.exists())

### Step 2 — Submit an STT request

Unlike TTS where we sent text, here we upload an audio file. We can optionally enable speaker identification (diarisation).

In [ ]:
with open(AUDIO_PATH, "rb") as f:
    stt_response = requests.post(
        f"{BASE_URL}/api/stt/{session_id}",
        files={"audio": f},
        data={"enable_speaker_identification": "true"}
    ).json()

request_id = stt_response["request_id"]
print("STT request_id:", request_id)

### Step 3 — Poll for the result

Same pattern as notebook 01 — keep checking until `"completed"`.

In [ ]:
MAX_POLLS = 120

for _ in range(MAX_POLLS):
    result = requests.get(f"{BASE_URL}/api/result/{request_id}").json()
    if result["status"] == "completed":
        break
    time.sleep(1)
else:
    raise TimeoutError("STT request did not complete in time.")

result.keys()

### Step 4 — Explore the transcription

The result contains top-level metadata plus detailed `segments` and `words` arrays. Let's print the highlights.

In [ ]:
transcription = result["result"]

print("Text:", transcription["text"])
print("Duration:", transcription["duration"])
print("Language:", transcription["language"])
print("Total words:", transcription["total_words"])
print("Total segments:", transcription["total_segments"])

### Step 5 — View as DataFrames

Putting the segments and words into DataFrames makes them easier to browse and filter. Segments are larger chunks of speech; words give you individual word-level timing and confidence.

In [ ]:
segments_df = pd.DataFrame(transcription["segments"])
segments_df

In [ ]:
words_df = pd.DataFrame(transcription["words"])
words_df.head(20)

## Try it yourself

**Easy:** Re-run with `enable_speaker_identification` set to `"false"` — what changes in the output?

**Medium:** Filter `words_df` to only show words with a confidence score below 0.8. Are there patterns in what the model is less sure about?

**Stretch:** Use the TTS from notebook 01 to generate a WAV file, then transcribe it here with STT. Does the transcription match the original text?